1. Используйте подход Stacking для решения ML задачи на любом выбранном датасете (в качестве мета модели выберете LogisticRegression)
2. Примените любой подход для интерпретации получившегося ML алгоритма (оцените важность признаков)

In [ ]:
!pip install -qU shap lime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from lime.lime_tabular import LimeTabularExplainer
from sklearn.inspection import permutation_importance
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

In [ ]:
(X_train, y_train), (X_test, y_test) = mnist.load_data()

# Нормализация и выравнивание в векторы
X_train = X_train.reshape(X_train.shape[0], -1) / 255.0
X_test = X_test.reshape(X_test.shape[0], -1) / 255.0

# Подвыборка для обучения
train_idx = np.random.choice(len(X_train), 5000, replace=False)
test_idx = np.random.choice(len(X_test), 500, replace=False)

X_train_s, y_train_s = X_train[train_idx], y_train[train_idx]
X_test_s, y_test_s = X_test[test_idx], y_test[test_idx]

NameError: name 'mnist' is not defined

In [ ]:
base_estimators = [
    ('rf', RandomForestClassifier(n_estimators=50, random_state=42)),
    ('knn', KNeighborsClassifier(n_neighbors=5))
]

stacking_model = StackingClassifier(
    estimators=base_estimators,
    final_estimator=LogisticRegression(max_iter=1000),
    cv=3
)

stacking_model.fit(X_train_s, y_train_s)
print(f"Model Accuracy: {accuracy_score(y_test_s, stacking_model.predict(X_test_s)):.4f}")

In [ ]:
y_pred = stacking_model.predict(X_test_s)
print(f"Accuracy: {accuracy_score(y_test_s, y_pred):.4f}")
print(classification_report(y_test_s, y_pred))

In [ ]:
# Оцениваем важность базовых моделей
perm_results = permutation_importance(
    stacking_model, X_test_s, y_test_s,
    n_repeats=5, random_state=42, n_jobs=-1
)

feature_names = []
for name, _ in base_estimators:
    for i in range(10): feature_names.append(f"{name}_class_{i}")

perm_data = pd.DataFrame(
    stacking_model.final_estimator_.coef_.mean(axis=0),
    index=feature_names,
    columns=['importance']
).sort_values(by='importance', ascending=False)

perm_data.plot(kind='barh', figsize=(10, 6), color='skyblue')
plt.title("Важность прогнозов базовых моделей (на основе весов мета-модели)")
plt.show()

In [ ]:
# Входные данные для мета-модели — это вероятности 10 классов от 2-х моделей (всего 20 признаков)
feature_names = []
for name, _ in base_estimators:
    for i in range(10):
        feature_names.append(f"{name}_class_{i}")

# Получаем абсолютные значения коэффициентов (важность)
importance = np.abs(stacking_model.final_estimator_.coef_).mean(axis=0)

perm_data = pd.DataFrame({
    'feature': feature_names,
    'importance': importance
}).sort_values(by='importance', ascending=True)

plt.figure(figsize=(10, 8))
plt.barh(perm_data['feature'], perm_data['importance'], color='teal')
plt.title("Глобальная важность предсказаний базовых моделей (Coefficients)")
plt.xlabel("Средний абсолютный вес в логистической регрессии")
plt.tight_layout()
plt.show()

In [ ]:
import shap
import numpy as np

# 1. Готовим данные
# Получаем "признаки" для мета-модели (вероятности от базовых моделей)
X_meta = stacking_model.transform(X_test_s)
# Считаем среднее значение каждого признака (базовый фон)
X_mean = X_meta.mean(axis=0)

# 2. Выбираем объект для объяснения
idx = 0
pred_class = int(stacking_model.predict(X_test_s[idx:idx+1])[0])

# 3. Достаем веса конкретного класса из логистической регрессии
# coef_ имеет форму (10, 20), берем строку для нашего класса
coef = stacking_model.final_estimator_.coef_[pred_class]
intercept = stacking_model.final_estimator_.intercept_[pred_class]

# 4. Считаем SHAP values
features_val = X_meta[idx]
shap_values_manual = (features_val - X_mean) * coef
base_value = np.dot(X_mean, coef) + intercept

# 5. Готовим имена признаков
feature_names = []
for name, _ in base_estimators:
    for i in range(10): feature_names.append(f"{name}_{i}")

print(f"Объяснение для объекта №{idx} (Цифра {y_test_s[idx]})")
print(f"Предсказано: {pred_class}")

# 6. Рисуем график
shap.initjs()

# Принудительно выравниваем размерности (на всякий случай)
shap.force_plot(
    base_value=base_value,
    shap_values=shap_values_manual.flatten(),
    features=features_val.flatten(),
    feature_names=feature_names
)

In [ ]:
from lime import lime_image
from skimage.segmentation import mark_boundaries

explainer_lime = lime_image.LimeImageExplainer()

# Функция, которая превращает картинку (28,28,3) обратно в вектор (784,) для модели
def predict_wrapper(images):
    flat_images = [img[:,:,0].flatten() for img in images]
    return stacking_model.predict_proba(flat_images)

# Берем тестовое изображение и делаем его "цветным" (требование LIME)
img_idx = 0
img_rgb = np.stack([X_test_s[img_idx].reshape(28,28)]*3, axis=-1)

# Запускаем объяснение
explanation = explainer_lime.explain_instance(
    img_rgb,
    predict_wrapper,
    top_labels=5,
    hide_color=0,
    num_samples=500  # Можно уменьшить до 100, если будет долго
)

# Визуализируем
temp, mask = explanation.get_image_and_mask(
    pred_class, positive_only=True, num_features=10, hide_rest=False
)

plt.figure(figsize=(6,6))
plt.imshow(mark_boundaries(temp, mask))
plt.title(f"LIME: Самые важные области для цифры {pred_class}")
plt.axis('off')
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Получаем предсказания
y_pred = stacking_model.predict(X_test_s)

# 2. Считаем матрицу
cm = confusion_matrix(y_test_s, y_pred)

# 3. Рисуем
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title("Матрица ошибок (Confusion Matrix)")
plt.xlabel("Предсказание модели")
plt.ylabel("Истинный класс")
plt.show()